In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
# Device configuration (use GPU if available)
device = torch.device("mps")


In [2]:
class KeypointDataset(Dataset):
    def __init__(self, csv_file):
        data = pd.read_csv(csv_file)
        
        # Separate features and target
        self.X = data.iloc[:, :-1].values  # All keypoint columns
        self.y = data.iloc[:, -1].values   # Class column
        
        # Encode target labels into integers
        self.le = LabelEncoder()
        self.y = self.le.fit_transform(self.y)
        
        # Convert to tensor
        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [3]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out


In [8]:
# Hyperparameters
input_size = 42  # 20 keypoints, each with x and y coordinates
hidden_size = 128  # Number of neurons in hidden layers
num_classes = 26  # Letters A-Z
batch_size = 32
num_epochs = 30
learning_rate = 0.001

train_dataset = KeypointDataset('data/train.csv')
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = KeypointDataset('data/val.csv')
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Initialize the MLP model, loss function, and optimizer
model = MLP(input_size, hidden_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [9]:
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in tqdm(enumerate(train_loader), desc=f'Epoch {epoch+1}/{num_epochs}', total=len(train_loader)):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')


Epoch 1/50: 100%|██████████| 93/93 [00:01<00:00, 80.76it/s] 


Epoch [1/50], Loss: 2.7613


Epoch 2/50: 100%|██████████| 93/93 [00:00<00:00, 202.46it/s]


Epoch [2/50], Loss: 1.5201


Epoch 3/50: 100%|██████████| 93/93 [00:00<00:00, 198.92it/s]


...

Epoch [48/50], Loss: 0.1546


Epoch 49/50: 100%|██████████| 93/93 [00:00<00:00, 207.12it/s]


Epoch [49/50], Loss: 0.1442


Epoch 50/50: 100%|██████████| 93/93 [00:00<00:00, 210.28it/s]

Epoch [50/50], Loss: 0.1500


In [10]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 91.53%


In [9]:

# Load the test dataset (with labels)
test_dataset = KeypointDataset('data2/val.csv')
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [11]:
model.eval()  # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad():  # No need to compute gradients for evaluation
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to the same device
        outputs = model(inputs)  # Forward pass
        _, predicted = torch.max(outputs.data, 1)  # Get predictions from the output
        total += labels.size(0)  # Total number of test samples
        correct += (predicted == labels).sum().item()  # Count correctly predicted labels

# Calculate accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

Test Accuracy: 91.53%


In [12]:
example_input = torch.randn(1, input_size).to(device)  # Example input tensor of the same shape as the data
traced_script_module = torch.jit.trace(model, example_input)

# Save the traced model
traced_script_module.save("hand_keypoints_classifier_new.pt")
print("Model saved in TorchScript format.")

Model saved in TorchScript format.


In [3]:
loaded_model = torch.jit.load("models/hand_keypoints_classifier_new_cpu.pt")
loaded_model = loaded_model.to(device)  # Move model to the appropriate device (GPU or CPU)
print("TorchScript model loaded.")

TorchScript model loaded.


In [14]:
# Set model to evaluation mode
loaded_model.eval()

# Evaluate accuracy on the test set using the loaded model
correct = 0
total = 0

with torch.no_grad():  # No need to compute gradients for inference
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to the same device
        outputs = loaded_model(inputs)  # Forward pass through the loaded model
        _, predicted = torch.max(outputs.data, 1)  # Get predictions from the output
        total += labels.size(0)  # Total number of test samples
        correct += (predicted == labels).sum().item()  # Count correctly predicted labels

# Calculate and print the accuracy
accuracy = 100 * correct / total
print(f'Test Accuracy with TorchScript model: {accuracy:.2f}%')

Test Accuracy with TorchScript model: 91.53%
